# XAI Notebook

In [ ]:
!pip install lime shap dice-ml

# 1 Load and Preprocess Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

url = 'http://apmonitor.com/pds/uploads/Main/steel.txt'
df = pd.read_csv(url)

# The last 7 columns are the fault types (Binary encoded)
# We convert them back to a single 'FaultType' label for classification
fault_columns = ["Pastry", "Z_Scratch", "K_Scatch", "Stains", "Dirtiness", "Bumps", "Other_Faults"]
df['FaultType'] = df[fault_columns].idxmax(axis=1)

# Drop the individual binary fault columns and set up Features/Target
X = df.drop(columns=fault_columns + ['FaultType'])
y = df['FaultType']

# Numerically encode the target variable for models and DiCE
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(le.classes_)
pd.concat([X,pd.Series(y_encoded, name="Encoded Fault")], axis=1)

# 2. Train Model

In [ ]:
X_train, X_test, y_train_encoded, y_test_encoded = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=500, random_state=42)
clf.fit(X_train, y_train_encoded)

# 3. Generate LIME Explanation

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X.columns.tolist(),
    class_names=le.classes_.tolist(), # Use the original string class names for LIME display
    mode='classification'
)

# Explain a specific defect
# Let's find a case of a "Bumps" defect to analyze from the test set
# Get the encoded value for 'Bumps'
bumps_encoded_value = le.transform(['Bumps'])[0]

# Find the positional indices in y_test_encoded that correspond to 'Bumps'
test_bumps_pos_indices = np.where(y_test_encoded == bumps_encoded_value)[0]

# Define a global variable to store the chosen instance's positional index in the test set
global explanation_instance_idx_in_test

if len(test_bumps_pos_indices) == 0:
    print("No 'Bumps' defects found in the test set. Explaining the first instance in X_test.")
    explanation_instance_idx_in_test = 0 # Explain the first instance in X_test
else:
    explanation_instance_idx_in_test = test_bumps_pos_indices[0] # Get the first positional index of 'Bumps' in the test set

# Get the original true fault label string for the selected instance
true_fault_label_str = le.inverse_transform([y_test_encoded[explanation_instance_idx_in_test]])[0]
label_index = y_test_encoded[explanation_instance_idx_in_test]

sample_instance_for_lime = X_test.iloc[[explanation_instance_idx_in_test]]

exp = explainer.explain_instance(
    sample_instance_for_lime.values[0],
    clf.predict_proba,
    num_features=6,
    labels=[label_index]
)

print(f"True Fault: {true_fault_label_str}")
predicted_fault_encoded = clf.predict(sample_instance_for_lime)[0]
predicted_fault_str = le.inverse_transform([predicted_fault_encoded])[0]
print(f"Predicted Fault: {predicted_fault_str}")

exp.show_in_notebook(show_all=False)

## 5. Counterfactual Explanations with DiCE

### Set up DiCE Explainer

In [ ]:
import dice_ml

# Create a combined dataframe for DiCE, including features and the encoded target variable
df_train = pd.concat([X_train, pd.Series(y_train_encoded, index=X_train.index, name='FaultType_encoded')], axis=1)

# Define continuous and categorical features for DiCE
# All features in X appear to be numerical, so categorical_features will be empty.
dice_continuous_features = X.columns.tolist()
dice_categorical_features = []

# Create a DiCE data object
d = dice_ml.Data(
    dataframe=df_train,
    continuous_features=dice_continuous_features,
    categorical_features=dice_categorical_features,
    outcome_name='FaultType_encoded'
)

# Create a DiCE model object
m = dice_ml.Model(model=clf, backend='sklearn', model_type='classifier')

# Create a DiCE explainer object
exp_dice = dice_ml.Dice(d, m, method='random')

print("DiCE Explainer initialized successfully.")

### Generate and Display Counterfactuals

In [ ]:
# Use the same sample instance as for LIME explanation, referring to the global variable
sample_instance = X_test.iloc[[explanation_instance_idx_in_test]]

# Predict the fault for the sample instance (will be numerically encoded)
current_fault_prediction_encoded = clf.predict(sample_instance)[0]
current_fault_prediction_str = le.inverse_transform([current_fault_prediction_encoded])[0]

# Reuse true_fault_label_str from the previous cell for consistency
print(f"Original Instance True Fault: {true_fault_label_str}")
print(f"Original Instance Predicted Fault: {current_fault_prediction_str}")

# Let's try to find counterfactuals that change the predicted class
# We need to work with encoded class labels for DiCE.

# Choose a target class different from the current prediction (using encoded values)
all_encoded_classes = clf.classes_.tolist()
possible_target_classes_encoded = [c for c in all_encoded_classes if c != current_fault_prediction_encoded]

if possible_target_classes_encoded:
    target_class_encoded = possible_target_classes_encoded[0] # Pick the first available different encoded class
    target_class_str = le.inverse_transform([target_class_encoded])[0]
    print(f"\nGenerating counterfactuals to achieve target class: {target_class_str} (encoded: {target_class_encoded})")

    cf_explanation = exp_dice.generate_counterfactuals(
        query_instances=sample_instance,
        total_CFs=5,  # Number of counterfactuals to generate
        desired_class=target_class_encoded # Pass the encoded target class
    )

    # Visualize the counterfactuals
    cf_explanation.visualize_as_dataframe(show_only_changes=True) # show_only_changes for brevity
else:
    print("Cannot generate counterfactuals as there are no other target classes to aim for.")

## 6. SHAP (SHapley Additive exPlanations)
SHAP values attribute the difference between the actual prediction and the average prediction to each feature.

In [ ]:
import shap

# Initialize JavaScript for SHAP visualizations
shap.initjs()

# Create a SHAP explainer for the Tree-based model (Random Forest)
# We use the training data as a background for calculating expected values
explainer_shap = shap.TreeExplainer(clf)

# Calculate SHAP values for the test set
# This might take a moment depending on the size of the test set
shap_values = explainer_shap.shap_values(X_test)

print("SHAP values calculated successfully.")

### Global Interpretability: Summary Plot
This plot shows the most important features across all classes.

In [ ]:
# Summary plot for all classes
shap.summary_plot(shap_values, X_test, plot_type="bar", class_names=le.classes_)

### Local Interpretability
Let's look at the same instance we used for LIME to see how SHAP explains the prediction.

In [ ]:
# Identify the predicted class for the sample instance
predicted_class_idx = clf.predict(sample_instance)[0]
print(f"Explaining prediction for class: {le.inverse_transform([predicted_class_idx])[0]}")

if isinstance(shap_values, list):
    # List of [instance, feature]
    instance_shap_values = shap_values[predicted_class_idx][explanation_instance_idx_in_test, :]
else:
    # Array of [instance, feature, class]
    instance_shap_values = shap_values[explanation_instance_idx_in_test, :, predicted_class_idx]

instance_features = X_test.iloc[explanation_instance_idx_in_test, :]

print(f"SHAP values shape: {instance_shap_values.shape}")
print(f"Features shape: {instance_features.shape}")

# Generate the force plot
shap.force_plot(
    explainer_shap.expected_value[predicted_class_idx],
    instance_shap_values,
    instance_features,
    matplotlib=True
)

exp = shap.Explanation(
    values=instance_shap_values,
    base_values=explainer_shap.expected_value[predicted_class_idx],
    data=instance_features.values,
    feature_names=X.columns.tolist()
)

# Generate the waterfall plot
shap.plots.waterfall(exp)

## 7. Permutation Feature Importance
Permutation importance measures the increase in the prediction error of the model after we permute the feature's values, which breaks the relationship between the feature and the true outcome.

In [ ]:
from sklearn.inspection import permutation_importance
import pandas as pd
import matplotlib.pyplot as plt

# Calculate permutation importance on the test set
# n_repeats is the number of times a feature is permuted to get a stable estimate
result = permutation_importance(
    clf, X_test, y_test_encoded, n_repeats=10, random_state=42, n_jobs=-1
)

# Organize the results into a DataFrame
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std
}).sort_values(by='importance_mean', ascending=False)

# Plot the top 15 results
plt.figure(figsize=(10, 8))
plt.barh(importance_df['feature'].head(15), importance_df['importance_mean'].head(15), xerr=importance_df['importance_std'].head(15))
plt.gca().invert_yaxis()
plt.title('Top 15 Features - Permutation Importance (Scikit-Learn)')
plt.xlabel('Importance (Decrease in Accuracy)')
plt.tight_layout()
plt.show()

display(importance_df.head(10))

## 8. Partial Dependence Plots (PDP)
Partial Dependence Plots show the marginal effect one or two features have on the predicted outcome of a machine learning model. They tell us how the model's prediction changes as a feature's value changes, averaging out the effects of all other features.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# Choose a target class to visualize (e.g., 'Bumps' or 'Pastry')
target_class_name = le.classes_[0]
target_class_idx = 0

# Convert training data to float to avoid 'LossySetitemError' in PDP
# This resolves the TypeError when trying to put floats into int64 columns
X_train_float = X_train.astype(float)

print(f"Generating PDP for features: {top_features}")
print(f"Target class: {target_class_name}")

# Create the PDP plots
fig, ax = plt.subplots(figsize=(12, 10))
display_pdp = PartialDependenceDisplay.from_estimator(
    clf,
    X_train_float,
    features=top_features,
    kind='average',
    target=target_class_idx,
    grid_resolution=50,
    ax=ax
)

plt.suptitle(f'Partial Dependence Plots for {target_class_name}')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 9. 2-Way Partial Dependence Plot
This plot shows the interaction between 'Length_of_Conveyer' and 'Steel_Plate_Thickness' and their joint effect on the predicted probability of the 'Bumps' class.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# Define the interaction pair
interaction_features = [('Length_of_Conveyer', 'Steel_Plate_Thickness')]

# Create the plot
fig, ax = plt.subplots(figsize=(10, 8))
display_2way = PartialDependenceDisplay.from_estimator(
    clf,
    X_train_float,
    features=interaction_features,
    kind='average',
    target=target_class_idx,
    grid_resolution=50,
    ax=ax
)

plt.suptitle(f'2-Way Partial Dependence Plot: Interaction Effect for {target_class_name}')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()